# G2 — Với M = 4, ứng viên đứng đầu của R̄σ có ổn định không? (SD 1.5, 553 prompt, M = 64)

**Notebook này không sinh ảnh mới, không chạy Phase 2.** Mỗi ứng viên của pool seed 100 được chấm với **64 mẫu nhiễu i.i.d.** (σ = 1, clip về [−1, 1], như Phase 1). 64 mẫu chia thành 16 nhóm 4 mẫu, mỗi nhóm là "một lần chạy RS-LiDAR với M = 4".

**Mục đích (mock review 27/11, R1).** R1 cho rằng cận của Proposition 3 vô nghĩa ở M = 4, nên RS-LiDAR có thể chỉ chọn ngẫu nhiên trong nhóm đầu. Notebook đo trực tiếp: tỉ lệ top-1 của M = 4 trùng top-1 của M = 64, tỉ lệ nằm trong top-3 của M = 64, Kendall τ giữa hai thứ hạng, khoảng cách giữa hai ứng viên đầu so với sai số chuẩn của ước lượng M = 4, và tỉ lệ ứng viên B của Table 1 trùng top-1 của M = 64.

**Cách chạy:** chọn **A100** (L4 được nhưng ~3 lần lâu hơn), rồi **Runtime → Run all**. Khoảng **3–4 h trên A100**. Prompt chạy theo một thứ tự ngẫu nhiên cố định và điểm lưu theo prompt: **bị ngắt thì Run all lại**, phần đã chạy được giữ, và phần đã có vẫn là mẫu ngẫu nhiên của 553 prompt.

**Kết quả** (`MyDrive/RS-LiDAR/extra_experiments/G2_rank_stability/`): `rank_stability_summary.json`, `rank_stability_per_prompt.csv`, `scores/*.npy`, `G2_report.md`.

## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 (hoặc L4) GPU** và bật **High-RAM**.

In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    else:
        print("⚠️ Chú ý: GPU dưới 22GB VRAM, code sẽ tự động dùng chunking để bảo vệ bộ nhớ.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi

## 2. Gắn Kết Google Drive & Thiết Lập Lưu Trữ Tức Thì (Real-time Persistence)
Toàn bộ kết quả sẽ được ghi trực tiếp vào `My Drive/RS-LiDAR/` thông qua symlink. Sau mỗi prompt hoàn thành, file `.png`, `latent.pt` và `results.json` được lưu vĩnh viễn trên Drive.

In [ ]:
import os, shutil, subprocess
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)

# 2. Clone hoặc kéo cập nhật repo RS-LiDAR tự động
REPO_DIR = "/content/RS-LiDAR"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/leekwanreal/RS-LiDAR.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/lookahead_sampling.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}

# 3. Tạo Symbolic Links an toàn (Tuyệt đối không xóa dữ liệu cũ, tự động migrate nếu có file cục bộ)
for folder_name in ["Lookahead_samples", "Target_samples"]:
    link_path = os.path.join(WORKDIR, folder_name)
    drive_target = os.path.join(DRIVE_DIR, folder_name)
    os.makedirs(drive_target, exist_ok=True)

    if os.path.islink(link_path):
        if not os.path.exists(link_path):
            os.remove(link_path)
            os.symlink(drive_target, link_path)
    elif os.path.exists(link_path):
        print(f"📦 Phát hiện thư mục cục bộ {folder_name}, đang đồng bộ an toàn sang Google Drive...")
        for item in os.listdir(link_path):
            src_item = os.path.join(link_path, item)
            dst_item = os.path.join(drive_target, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        shutil.rmtree(link_path)
        os.symlink(drive_target, link_path)
    else:
        os.symlink(drive_target, link_path)

print("✅ Hệ thống Symlink đã sẵn sàng 100%! Toàn bộ dữ liệu được bảo lưu vĩnh viễn trên Drive:")
print("  📁", DRIVE_DIR)

## 3. Cài Đặt Môi Trường Chuẩn Xác (100% Trơn Tru & Không Xung Đột)
Cài đặt bộ thư viện đồng bộ, tắt backend TensorFlow để chống xung đột Protobuf, và tải vocab HPSv2.

In [ ]:
import os, urllib.request

# 1. Tắt backend TensorFlow để loại bỏ xung đột Protobuf runtime_version
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 2. Nâng cấp protobuf và cài đặt các thư viện tương thích PyTorch 2.x
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q image-reward || pip install -q --no-deps git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Đảm bảo file từ điển OpenCLIP cho HPSv2
import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    print("📥 Đang tải file BPE vocab cho HPSv2...")
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("✅ Cài đặt môi trường hoàn tất 100%!")

In [ ]:
# Kiểm tra môi trường sau khi cài: in phiên bản và dừng ngay nếu thiếu thư viện chấm điểm
# (thiếu hpsv2/clip thì code reward giờ báo lỗi thay vì chấm 0; dừng ở đây đỡ tốn GPU hơn).
import importlib, sys
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
from rslidar_utils import setup_runtime
setup_runtime()  # cùng các shim tương thích mà script của repo áp dụng trước khi import ImageReward
import torch, numpy, transformers, diffusers
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__,
      "| transformers", transformers.__version__, "| diffusers", diffusers.__version__)
for mod in ("hpsv2", "clip", "open_clip", "ImageReward", "timm", "google.protobuf"):
    try:
        m = importlib.import_module(mod)
        print(f"  ✅ {mod} {getattr(m, '__version__', '')}")
    except Exception as e:
        raise RuntimeError(f"❌ Không import được {mod}: {e}. Chạy lại cell cài đặt (hoặc Runtime -> Restart session rồi chạy lại).")

## 4. Cấu hình chung (giống Table 1) và hàm tiện ích

In [ ]:
import os, sys, glob, json, shutil, subprocess, random, math
import numpy as np, pandas as pd
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
%cd {WORKDIR}
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

EXP_NAME = "G2_rank_stability"
EXP_DRIVE = f"{DRIVE_DIR}/extra_experiments/{EXP_NAME}"      # bảng tổng hợp CSV/JSON lưu vĩnh viễn trên Drive
LOCAL_DERIVED = f"/content/derived_lookahead/{EXP_NAME}"    # lookahead dẫn xuất: tạo lại được, để trên đĩa Colab
os.makedirs(EXP_DRIVE, exist_ok=True)
os.makedirs(LOCAL_DERIVED, exist_ok=True)

# ---- Cấu hình Table 1: SD 1.5, lookahead DPM-5 với K = 50, target DDPM-100, seed 100 ----
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
MODEL_TAG = "SD15"
STEPS_P1, K = 5, 50
SEED_P1, SEED_P2 = 100, 100
SCALE, STEPS_P2, ETA_P2 = 12.5, 100, 1.0
NUM_PARTICLES_P2, RESAMPLE_T_END = 4, 200
REF_LAMBDA = 5000
RS_SIGMA, RS_M = 1.0, 4          # RS-LiDAR DDPM của Table 1
RS_USE_CRN = False               # bài ghi "Perturbations: i.i.d." -> chỉ nhận pool/run I.I.D. (tên không có _crn)

PROMPT_MODE = "ALL"   # "ALL" (553 prompt) hoặc "100_REPRESENTATIVE" (chạy thử nhanh)
if PROMPT_MODE == "100_REPRESENTATIVE":
    MAX_PROMPT, PROMPT_INDICES_ARG, SUBSET_TAG = 100, "stratified_100", "_100p"
else:
    MAX_PROMPT, PROMPT_INDICES_ARG, SUBSET_TAG = 553, None, ""

from rslidar_utils import parse_prompt_indices
META = [json.loads(l) for l in open(f"{WORKDIR}/prompt_files/geneval_metadata.jsonl", encoding="utf-8") if l.strip()]
PROMPT_IDS = list(parse_prompt_indices(PROMPT_INDICES_ARG) or range(len(META)))[:MAX_PROMPT]
print(f"{EXP_NAME}: {len(PROMPT_IDS)} prompt | kết quả tổng hợp lưu tại {EXP_DRIVE}")

In [ ]:
import torch
from tqdm.auto import tqdm
from PIL import Image

STEP_VARIANTS = [f"Step{STEPS_P1}", f"DPM{STEPS_P1}", f"DPM{STEPS_P1}_Step{STEPS_P1}"]


def n_have(d, ids=None):
    """Số prompt trong PROMPT_IDS đã có results.json trong thư mục d."""
    ids = PROMPT_IDS if ids is None else ids
    if not d or not os.path.isdir(d):
        return 0
    return sum(os.path.exists(f"{d}/{i:05d}/results.json") for i in ids)


def find_lookahead(names):
    """Thư mục Lookahead đầu tiên phủ đủ PROMPT_IDS (thư mục 553 prompt cũng dùng được cho tập 100)."""
    for name in names:
        for suf in dict.fromkeys([SUBSET_TAG, ""]):
            p = f"{WORKDIR}/Lookahead_samples/{name}{suf}"
            if n_have(p) >= len(PROMPT_IDS):
                return p
    return None


def find_target(names):
    """Thư mục Target theo quy ước tên của notebook Table 1 (..._seed100[_100p]_A100)."""
    for name in names:
        # run mốc phải cùng tập prompt: seed toàn cục nối tiếp qua các prompt, nên run 553 prompt không ghép cặp với tập 100
        for suf in ([SUBSET_TAG] if SUBSET_TAG else [""]):
            for p in (f"{WORKDIR}/Target_samples/{name}{suf}_A100", f"{WORKDIR}/Target_samples/{name}{suf}"):
                if n_have(p) >= len(PROMPT_IDS):
                    return p
    return None


def raw_lookahead_names(seed=SEED_P1):
    names = [f"Lookahead_{MODEL_TAG}_{s}_n{K}_seed{seed}" for s in STEP_VARIANTS]
    names.append(f"{seed}_{K}_{STEPS_P1}")
    if seed == 100:
        names.append(f"100_{K}_{STEPS_P1}")
    return names


def rs_lookahead_names(sigma=RS_SIGMA, M=RS_M):
    crns = ["", "_crn"] if RS_USE_CRN is None else (["_crn"] if RS_USE_CRN else [""])
    return [f"RSLookahead_{MODEL_TAG}_{s}_n{K}_sig{sigma}_M{M}{c}_seed{SEED_P1}" for c in crns for s in STEP_VARIANTS]


def ref_target_names(rs):
    solver = f"DDPM{STEPS_P2}"
    if rs:
        crns = ["", "_crn"] if RS_USE_CRN is None else (["_crn"] if RS_USE_CRN else [""])
        return [f"RSLiDAR_{MODEL_TAG}_{s}_n{K}_sig{RS_SIGMA}_M{RS_M}{c}_{solver}_s{SCALE}_lmbda{REF_LAMBDA}_seed{SEED_P2}"
                for c in crns for s in STEP_VARIANTS]
    return [f"LiDAR_{MODEL_TAG}_{s}_n{K}_{solver}_s{SCALE}_lmbda{REF_LAMBDA}_seed{SEED_P2}" for s in STEP_VARIANTS]


def stream(cmd):
    print("$ " + " \\\n    ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))  # log hiện ngay, không bị giữ trong bộ đệm
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {cmd[1]}")


def phase1(run_name, seed=SEED_P1, reuse_from=None, smoothing=False, sigma=RS_SIGMA, M=RS_M, crn=False,
           metrics="ImageReward"):
    """Phase 1 (lookahead_sampling.py) với đúng các cờ của notebook Table 1; bỏ qua nếu đã đủ."""
    out = f"{WORKDIR}/Lookahead_samples/{run_name}"
    if n_have(out) >= len(PROMPT_IDS):
        print(f"✅ Phase 1 đã đủ {len(PROMPT_IDS)} prompt: {run_name}")
        return out
    cmd = ["python", "lookahead_sampling.py", f"--seed={seed}", f"--num_particles={K}",
           f"--num_inference_steps={STEPS_P1}", f"--model_name={MODEL_NAME}",
           "--guidance_reward_fn=ImageReward", f"--metrics_to_compute={metrics}",
           "--prompt_path=prompt_files/geneval_metadata.jsonl", f"--max_prompt={MAX_PROMPT}",
           f"--run_name={run_name}", "--resume"]
    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    if reuse_from:
        cmd.append(f"--reuse_latents_from={reuse_from}")
    if smoothing:
        cmd += ["--use_smoothing", f"--sigma={sigma}", f"--num_mc_samples={M}"]
        if crn:
            cmd.append("--use_crn")
    stream(cmd)
    return out


def phase2(lookahead_dir, run_name, lmbda=REF_LAMBDA, top_k=K, pps=False):
    """Phase 2 (LiDAR_sampling.py) với đúng các cờ của notebook Table 1; bỏ qua nếu đã đủ.

    pps=True thêm --per_prompt_seed (nhiễu theo prompt: chạy tiếp sau khi bị ngắt vẫn đúng cặp).
    pps=False (so với run Table 1, seed toàn cục): run dở dang bị chặn, phải xoá và chạy lại một mạch.
    """
    out = f"{WORKDIR}/Target_samples/{run_name}"
    have = n_have(out)
    if have >= len(PROMPT_IDS):
        print(f"✅ Phase 2 đã đủ {len(PROMPT_IDS)} prompt: {run_name}")
        return out
    if have > 0 and not pps:
        raise RuntimeError(f"Run dở dang ({have}/{len(PROMPT_IDS)} prompt): {out}\n"
                           "Run này dùng seed toàn cục nên chạy tiếp sẽ lệch nhiễu so với run đối chứng. "
                           "Xoá thư mục trên rồi chạy lại cell này.")
    cmd = ["python", "LiDAR_sampling.py", f"--seed={SEED_P2}", f"--model_name={MODEL_NAME}",
           f"--num_particles={NUM_PARTICLES_P2}", f"--num_inference_steps={STEPS_P2}", f"--eta={ETA_P2}",
           "--use_rag", f"--lookahead_path={lookahead_dir}", f"--top_k={top_k}", f"--scale={SCALE}",
           f"--lmbda={lmbda}", f"--resample_t_end={RESAMPLE_T_END}",
           "--prompt_path=prompt_files/geneval_metadata.jsonl", f"--max_prompt={MAX_PROMPT}",
           "--guidance_reward_fn=ImageReward",
           "--metrics_to_compute=ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS",
           "--save_individual_images", f"--run_name={run_name}", "--resume"]
    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    if pps:
        cmd.append("--per_prompt_seed")
    stream(cmd)
    return out


def pools_paired(a, b, n_check=3):
    """True if pools a and b hold the same latents (up to order) for the first n_check prompts."""
    for i in PROMPT_IDS[:n_check]:
        if not (os.path.exists(f"{a}/{i:05d}/samples/latent.pt") and os.path.exists(f"{b}/{i:05d}/samples/latent.pt")):
            return False
        La, _ = load_pool(a, i)
        Lb, _ = load_pool(b, i)
        if La.shape != Lb.shape:
            return False
        A, Bm = La.flatten(1).float(), Lb.flatten(1).float()
        d = torch.cdist(Bm, A, compute_mode="donot_use_mm_for_euclid_dist").min(1).values
        if (d > 1e-3 * A.norm(dim=1).mean()).any():
            return False
    return True


def paired_rs_pool(raw_dir, rs_dir):
    """RS pool scored on exactly raw_dir's latents. If the pool found on Drive was generated separately
    (Phase 1 without --reuse_latents_from draws different candidates), rescore raw_dir's latents instead."""
    if rs_dir and pools_paired(raw_dir, rs_dir):
        print(f"✅ Pool RS dùng chung latents với pool LiDAR: {os.path.basename(rs_dir)}")
        return rs_dir
    if rs_dir:
        print(f"⚠️ Pool RS '{os.path.basename(rs_dir)}' KHÔNG dùng chung latents với pool LiDAR -> chấm lại pool LiDAR bằng R̄σ.")
    name = f"RSLookahead_{MODEL_TAG}_Step{STEPS_P1}_n{K}_sig{RS_SIGMA}_M{RS_M}_seed{SEED_P1}_rescore{SUBSET_TAG}"
    return phase1(name, reuse_from=raw_dir, smoothing=True)


def load_pool(d, i, key="ImageReward"):
    """Latents (K, 4, 64, 64) và reward (K,) của prompt i trong thư mục lookahead d."""
    lat = torch.load(f"{d}/{i:05d}/samples/latent.pt", map_location="cpu")
    lat = torch.stack(list(lat)) if isinstance(lat, (list, tuple)) else lat
    res = json.load(open(f"{d}/{i:05d}/results.json"))
    return lat, np.asarray(res[key]["result"], dtype=np.float64)


def _summ(v):
    v = np.asarray(v, dtype=np.float64)
    return {"result": v.tolist(), "mean": float(v.mean()), "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
            "max": float(v.max()), "min": float(v.min())}


def write_derived(src_dir, dst_dir, transform):
    """Tạo thư mục lookahead dẫn xuất: cùng latents (symlink), reward mới = transform(i, results).

    LiDAR_sampling.py đọc reward từ khoá "ImageReward" của results.json, nên reward mới được ghi vào
    khoá đó; reward gốc giữ ở "source_results" để đối chiếu. Thứ tự latents không đổi.
    """
    n = 0
    for i in PROMPT_IDS:
        s, d = f"{src_dir}/{i:05d}", f"{dst_dir}/{i:05d}"
        if not os.path.exists(f"{s}/results.json"):
            continue
        res = json.load(open(f"{s}/results.json"))
        new = np.asarray(transform(i, res), dtype=np.float64)
        os.makedirs(f"{d}/samples", exist_ok=True)
        lat_dst = f"{d}/samples/latent.pt"
        if not os.path.lexists(lat_dst):
            os.symlink(f"{s}/samples/latent.pt", lat_dst)
        out = {"ImageReward": _summ(new), "prompt": res["prompt"], "prompt_index": i,
               "source_results": {k: v for k, v in res.items() if isinstance(v, dict) and "result" in v}}
        json.dump(out, open(f"{d}/results.json", "w"))
        n += 1
    print(f"📁 {os.path.basename(dst_dir)}: {n} prompt")
    return dst_dir


def rs_pool_ok(d, n_check=3):
    """Pool RS chỉ dùng được nếu được chấm đúng cấu hình bài: σ, M, nhiễu i.i.d., β = 0, làm trơn trên ảnh.

    Tên thư mục không ghi cách lấy nhiễu, nên phải đọc results.json["smoothing"]. Metadata có ba đời
    (rslidar_utils.read_smoothing_config): trước 27/10/2026 chỉ có use_crn (nhiễu i.i.d. hoặc CRN, không có
    entropic risk); 27/10 - 9/11 mặc định antithetic + β = 0.5; từ 9/11 mặc định i.i.d., β = 0.
    """
    from rslidar_utils import read_smoothing_config
    want = {"sigma": float(RS_SIGMA), "num_mc_samples": int(RS_M), "noise_mode": "independent_iid",
            "beta_risk": 0.0, "domain": "image"}
    legacy = False
    for i in PROMPT_IDS[:n_check]:
        f = f"{d}/{i:05d}/results.json"
        if not os.path.exists(f):
            return False
        sm = read_smoothing_config(json.load(open(f)))
        if not sm:
            print(f"⚠️ {os.path.basename(d)}: results.json không ghi cấu hình smoothing -> không dùng pool này.")
            return False
        bad = {k: (sm[k], v) for k, v in want.items()
               if (sm[k] != v if isinstance(v, str) else abs(float(sm[k]) - v) > 1e-9)}
        if bad:
            print(f"⚠️ {os.path.basename(d)}: cấu hình smoothing khác bài {bad} (có, cần) -> không dùng pool này.")
            return False
        legacy = legacy or sm["legacy_metadata"]
    if legacy:
        print(f"ℹ️ {os.path.basename(d)}: metadata đời cũ (trước 27/10/2026, chỉ có use_crn = False) "
              "-> nhiễu i.i.d., β = 0, đúng cấu hình bài.")
    return True


def ref_ok(target, pool, rs=False, lmbda=REF_LAMBDA, top_k=K):
    """Run mốc (Table 1) chỉ dùng được nếu args.json khớp cấu hình ghép cặp: cùng pool ứng viên, seed toàn cục,
    cùng λ, K, số bước, scale. Không khớp thì trả về False để notebook tự chạy lại mốc."""
    if not target:
        return False
    name = os.path.basename(os.path.normpath(target))
    f = f"{target}/args.json"
    if not os.path.exists(f):
        print(f"⚠️ {name}: không có args.json, không kiểm tra được cấu hình -> vẫn dùng.")
        return True
    a = json.load(open(f))
    want = {"seed": SEED_P2, "lmbda": lmbda, "top_k": top_k, "num_inference_steps": STEPS_P2,
            "num_particles": NUM_PARTICLES_P2, "scale": SCALE, "eta": ETA_P2}
    bad = {k: (a.get(k), v) for k, v in want.items() if k in a and abs(float(a[k]) - float(v)) > 1e-9}
    if a.get("per_prompt_seed"):
        bad["per_prompt_seed"] = (True, False)
    lp = str(a.get("lookahead_path", ""))
    ref_pool = (lp if os.path.isabs(lp) and os.path.exists(lp) else
                f"{WORKDIR}/Lookahead_samples/{os.path.basename(os.path.normpath(lp))}") if lp else None
    if rs and ref_pool and os.path.exists(ref_pool) and not rs_pool_ok(ref_pool):
        bad["smoothing của pool"] = (os.path.basename(os.path.normpath(ref_pool)), "i.i.d., σ, M như bài")
    if pool and ref_pool:
        same = os.path.basename(os.path.normpath(ref_pool)) == os.path.basename(os.path.normpath(pool))
        if not same:
            same = os.path.exists(ref_pool) and pools_paired(ref_pool, pool)
        if not same:
            bad["lookahead_path"] = (os.path.basename(os.path.normpath(lp)), os.path.basename(os.path.normpath(pool)))
    if bad:
        print(f"⚠️ Run mốc {name} không khớp cấu hình {bad} (có, cần) -> không dùng, notebook tự chạy lại mốc.")
        return False
    print(f"✅ Run mốc khớp cấu hình: {name}")
    return True

## 5. Pool LiDAR và pool RS (cùng latent)

In [ ]:
from rslidar_utils import read_smoothing_config

RAW_DIR = find_lookahead(raw_lookahead_names())
assert RAW_DIR, "Không tìm thấy pool lookahead LiDAR seed 100 (Table 1)."
assert not read_smoothing_config(json.load(open(f"{RAW_DIR}/{PROMPT_IDS[0]:05d}/results.json"))), \
    "RAW_DIR có cấu hình làm trơn -> không phải pool LiDAR"
print("✅ Pool LiDAR:", os.path.basename(RAW_DIR))
RS_DIR = find_lookahead(rs_lookahead_names())
assert RS_DIR and rs_pool_ok(RS_DIR), "Không tìm thấy pool RS σ = 1, M = 4, i.i.d. của Table 1."
assert pools_paired(RAW_DIR, RS_DIR), "Pool RS không dùng chung latent với pool LiDAR -> báo nhóm, không chạy tiếp."
print("✅ Pool RS (cùng latent):", os.path.basename(RS_DIR))

## 5b. Chép hai pool về ổ đĩa của máy Colab (song song, vài phút)
Đọc Drive từng file rất chậm (đo 28/11: ~0,7 MB/s, ~30 phút cho 553 prompt), nên chép một lần bằng nhiều luồng; mọi bước sau đọc bản chép. Chép tiếp được nếu bị ngắt. Điểm vẫn ghi lên Drive.

In [ ]:
from experiments import pool_tools as pt

LOCAL_DIR = "/content/local_cache"
DRIVE_RAW_DIR, DRIVE_RS_DIR = RAW_DIR, RS_DIR
RAW_DIR = pt.stage_local(DRIVE_RAW_DIR, f"{LOCAL_DIR}/{os.path.basename(DRIVE_RAW_DIR)}")
RS_DIR = pt.stage_local(DRIVE_RS_DIR, f"{LOCAL_DIR}/{os.path.basename(DRIVE_RS_DIR)}")
idx_both = set(pt.list_prompt_indices(RAW_DIR)) & set(pt.list_prompt_indices(RS_DIR))
assert set(PROMPT_IDS) <= idx_both, f"Chép thiếu: {len(idx_both)} prompt có đủ file ở cả hai pool"
print("✅ Các bước sau đọc từ ổ đĩa của máy:", RAW_DIR, RS_DIR, sep="\n  ")

## 6. Chạy thử 2 prompt với 8 mẫu (kiểm tra đường chạy)

In [ ]:
SMOKE = f"{EXP_DRIVE}/smoke"
stream(["python", "-m", "experiments.rank_stability", f"--raw_pool={RAW_DIR}", f"--rs_pool={RS_DIR}",
        f"--out_dir={SMOKE}", "--max_prompts=2", "--num_noise=8"])
print("✅ Chạy thử xong:", json.load(open(f"{SMOKE}/rank_stability_summary.json"))["n_prompts"], "prompt")

## 7. Chạy đầy đủ (553 prompt × 50 ứng viên × 64 mẫu), chạy tiếp được

In [ ]:
stream(["python", "-m", "experiments.rank_stability", f"--raw_pool={RAW_DIR}", f"--rs_pool={RS_DIR}",
        f"--out_dir={EXP_DRIVE}", "--resume"])

## 8. Báo cáo gửi lại

In [ ]:
s = json.load(open(f"{EXP_DRIVE}/rank_stability_summary.json"))
rows = [
    ("Số prompt", s["n_prompts"]),
    ("Top-1 của M = 4 trùng top-1 của M = 64 (trung bình)", s["agree_top1"]["mean"]),
    ("Top-1 của M = 4 nằm trong top-3 của M = 64", s["in_top3"]["mean"]),
    ("Tỉ lệ prompt mà cả 16 nhóm M = 4 cùng chọn một ứng viên", s["frac_prompts_all_groups_agree"]),
    ("Kendall τ (M = 4 so với M = 64)", s["kendall_tau"]["mean"]),
    ("P(đảo thứ tự hai ứng viên đầu) với M = 4", s["p_swap_top2"]["mean"]),
    ("Khoảng cách hai ứng viên đầu / sai số chuẩn M = 4 (trung vị)", s["gap_over_se"]["median"]),
    ("B của Table 1 trùng top-1 của M = 64", s.get("frac_B_is_top_all")),
    ("B của Table 1 nằm trong top-3 của M = 64", s.get("frac_B_in_top3")),
]
t = pd.DataFrame(rows, columns=["Chỉ số", "Giá trị"])
display(t)
agree = s["agree_top1"]["mean"]
note = ("Ổn định: M = 4 chọn gần như cùng ứng viên với M = 64." if agree >= 0.8 else
        "Ổn định vừa: báo đúng con số, giải thích trong D.8." if agree >= 0.6 else
        "Kém ổn định (< 60%): báo nhóm; có thể chạy Phase 2 với lựa chọn của M = 64 (tác giả quyết).")
text = "# G2: độ ổn định của ứng viên đứng đầu với M = 4 (SD 1.5, σ = 1, 64 mẫu)\n\n" + t.to_markdown(index=False) + f"\n\n{note}\n"
open(f"{EXP_DRIVE}/G2_report.md", "w", encoding="utf-8").write(text)
print(text)

## Tự ngắt máy khi chạy xong
Cell cuối tự ngắt runtime Colab sau khi Run all xong (tiết kiệm compute unit). Kết quả đã lưu trên Drive.

In [ ]:
# Tự ngắt runtime khi Run all chạy xong, để không tốn thêm compute unit.
# Kết quả đã nằm trên Drive; đẩy hết dữ liệu đang chờ ghi lên Drive trước khi ngắt.
# Muốn giữ máy để xem/chạy tiếp thì đặt AUTO_DISCONNECT = False trước khi Run all.
AUTO_DISCONNECT = True
if AUTO_DISCONNECT:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        print("🛑 Đã chạy xong và lưu lên Drive. Tự ngắt runtime Colab.")
        runtime.unassign()
    except Exception as e:
        print("⚠️ Không tự ngắt được runtime, hãy tự ngắt: Runtime → Disconnect and delete runtime.", e)